In [1]:
# Check required Python packages and install any that are missing or out of range.
import importlib.metadata
import subprocess
import sys
from packaging.requirements import Requirement

requirements = [
    "transformers<5.0",
    "huggingface-hub<1.0,>=0.34.0",
    "gradio<6",
    "numpy<2",
    "timm",
    "inflect",
    "phonemizer",
]

for requirement_text in requirements:
    requirement = Requirement(requirement_text)
    try:
        installed_version = importlib.metadata.version(requirement.name)
        satisfied = installed_version in requirement.specifier
    except importlib.metadata.PackageNotFoundError:
        installed_version = None
        satisfied = False

    if satisfied:
        print(f"{requirement.name}: already installed ({installed_version})")
    else:
        subprocess.check_call([sys.executable, "-m", "pip", "install", requirement_text])
        print(f"{requirement.name}: installed")


transformers: already installed (4.57.6)
huggingface-hub: already installed (0.36.2)
gradio: already installed (5.50.0)
numpy: already installed (1.26.4)
timm: already installed (1.0.28)
inflect: already installed (7.5.0)
phonemizer: already installed (3.3.0)


In [2]:
# Imports
from IPython.display import Audio as IPythonAudio
from IPython.display import Image as IPythonImage
from PIL import Image
import gradio as gr
from transformers import pipeline
from transformers.utils import logging
import numpy as np

from helper import (
    ignore_warnings,
    render_results_in_image,
    summarize_predictions_natural_language,
)

# Suppress warning messages
logging.set_verbosity_error()
ignore_warnings()

from phonemizer.backend.espeak.wrapper import EspeakWrapper

EspeakWrapper.set_library(
    r"C:\Program Files\eSpeak NG\libespeak-ng.dll"
)

In [9]:
# Import the model and create an instance
od_pipe = pipeline("object-detection", "facebook/detr-resnet-50")

# other model
# Text to Speech model
tts_pipe = pipeline("text-to-speech", model="kakao-enterprise/vits-ljs")


#from transformers import pipeline
#tts_pipe = pipeline("text-to-audio", model="suno/bark-small")
tts_pipe

c:\Users\ihurd\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\nn\modules\module.py:2588: UserWarning: for conv1.weight: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, which is a no-op. (Did you mean to pass `assign=True` to assign items in the state dictionary to their corresponding key in the module instead of copying them in place?)
  module._load_from_state_dict(
c:\Users\ihurd\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\nn\modules\batchnorm.py:141: UserWarning: for bn1.weight: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, which is a no-op. (Did you mean to pass `assign=True` to assign items in the state dictionary to their corresponding key in the module instead of copying them in place?)
  super()._load_from_state_dict(
c:\Users\ihurd\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\nn\modules\batchnorm.py:141: UserWarning: for bn1.bias

### Using `Gradio` to create a Simple Interface

Gradio is a Python library that simplifies the process of creating web interfaces for machine learning models. It allows you to quickly build interactive web-based applications where users can interact with your machine learning models without needing to write any HTML, CSS, or JavaScript code.

Once you've imported Gradio, you can use its functions and classes to define input and output components, specify your machine learning model, and create a web interface that integrates with your model.

In [10]:
# this function will do all the steps under the hood

#updating the pipelint to do the audio as well in the gradio app.
# detect, annotate, smmarize and generate audio summary.

def get_pipeline_prediction(pil_image):

    if pil_image is None:
        return None, "Please upload image!", None

    # this will process the image and identify the objects
    predictions = od_pipe(pil_image)

    # This will output the labelled image with boxes and confidence score
    processed_image = render_results_in_image(pil_image, predictions)

    #conver to labels , text.
    description = summarize_predictions_natural_language(predictions)

    #convert audio
    narrated = tts_pipe(description)

    # *** gradio audio shape, 1d (1, ssamples)
    audio = np.asarray(narrated["audio"]).squeeze()
    sampling = narrated["sampling_rate"]

    return processed_image,description, (sampling, audio)


In [ ]:
demo = gr.Interface(
  fn=get_pipeline_prediction,
  inputs=gr.Image(label="Input image",
                  type="pil"),
  outputs= [
              gr.Image(label="Output image with predicted instances", type="pil"),
              gr.Textbox(label="Image Description", lines = 3),
              gr.Audio(label = "Audio Description", autoplay = False)
  ],
  title = "ML2_8: Object Detection with Audio Assist",
  description = ("Upload an image to detect objects with bound boxes, "
                 "generate a descripion and playback in audio."
  )
)

# `share=True` will provide an online link to access to the demo'''
# share=True will create a public link for sharing
demo.launch(share=True)

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://e126a7d227f495f529.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


c:\Users\ihurd\AppData\Local\Programs\Python\Python313\Lib\site-packages\gradio\processing_utils.py:688: UserWarning: Trying to convert audio automatically from float32 to 16-bit int format.
  warnings.warn(warning.format(data.dtype))
c:\Users\ihurd\AppData\Local\Programs\Python\Python313\Lib\site-packages\gradio\processing_utils.py:688: UserWarning: Trying to convert audio automatically from float32 to 16-bit int format.
  warnings.warn(warning.format(data.dtype))
c:\Users\ihurd\AppData\Local\Programs\Python\Python313\Lib\site-packages\gradio\processing_utils.py:688: UserWarning: Trying to convert audio automatically from float32 to 16-bit int format.
  warnings.warn(warning.format(data.dtype))
c:\Users\ihurd\AppData\Local\Programs\Python\Python313\Lib\site-packages\gradio\processing_utils.py:688: UserWarning: Trying to convert audio automatically from float32 to 16-bit int format.
  warnings.warn(warning.format(data.dtype))


### Close the app
- Remember to call `.close()` on the Gradio app when you're done using it.

In [ ]:
# close when done
demo.close()

Closing server running on port: 7860
